# SQL4Business — Deliverable 4: fine-tuning del Planificador

Reproduce todo lo que reporta `docs/deliverable4.pdf`, en el hardware declarado (Colab T4):

1. `Runtime > Change runtime type > T4 GPU`, luego la celda de instalación.
2. **Datos** (sin GPU, segundos): regenera `data/finetune/` y comprueba que es idéntico al versionado.
3. **Entrenamiento** QLoRA del Planificador (unos 60–80 min) y publicación del adaptador en Hugging Face.
4. **Evaluación** de los cuatro sistemas y el control sobre las mismas 27 preguntas (unos 60 min).
5. **Demo del video**: Planificador base y Planificador ajustado sobre la misma entrada.

Para **solo reproducir los resultados** con el adaptador ya publicado, saltar el paso 3: la evaluación lo
descarga de Hugging Face (repositorio en `results/deliverable4/adapter_hf.json`).

In [ ]:
REPO_URL = "https://github.com/aredhel-jmze/Asistente-SQL-4-Business.git"
REF = "deliverable4"  # rama, tag o commit a evaluar

import os, subprocess, sys

os.chdir("/content")  # never stay inside a folder that a later cell may delete

ROOT = "/content/Asistente-SQL-4-Business"
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "--quiet", REPO_URL, ROOT], check=True)
subprocess.run(["git", "-C", ROOT, "fetch", "--quiet", "origin"], check=True)
# A branch is always taken at its latest version on GitHub; a tag or commit is used as is.
is_branch = subprocess.run(["git", "-C", ROOT, "rev-parse", "--verify", "--quiet", f"origin/{REF}"],
                           capture_output=True).returncode == 0
# --force discards files changed by an earlier run in the same Colab session (results, tables).
subprocess.run(["git", "-C", ROOT, "checkout", "--quiet", "--force", f"origin/{REF}" if is_branch else REF], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", f"{ROOT}/requirements.txt"], check=True)
os.chdir(ROOT)
print("commit:", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())

## 1. Datos de entrenamiento
Genera los pares (prompt del Planificador → plan correcto) y comprueba la separación con la evaluación (tests).

In [ ]:
!python scripts/build_planner_data.py | head -40
!git status --short data/finetune/   # vacío = idéntico a lo versionado
!python -m unittest tests.test_finetune_data -v 2>&1 | tail -12

## 2. Entrenamiento (QLoRA) y publicación

Iniciar sesión en Hugging Face con un token de **escritura** (solo para publicar). Cambiar `HF_REPO` por
la cuenta del equipo. El adaptador, `loss.csv` y `training_log.json` quedan en `results/deliverable4/adapter/`.

In [ ]:
from huggingface_hub import login
login()  # pega un token con permiso de escritura
HF_REPO = "CAMBIAR-usuario/sql4business-planificador-qlora"

In [ ]:
!python scripts/train_planner.py --limit 16 --epochs 1 --eval-every 1 --output-dir /content/prueba_adapter   # prueba corta (~3 min)

In [ ]:
!python scripts/train_planner.py --push-to-hub {HF_REPO}

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv("results/deliverable4/adapter/loss.csv")
ax = log.dropna(subset=["train_loss"]).plot(x="step", y="train_loss", alpha=.6)
log.dropna(subset=["dev_loss"]).plot(x="step", y="dev_loss", marker="o", ax=ax)
ax.set_ylabel("pérdida (entropía cruzada, tokens del plan)"); plt.show()

## 3. Evaluación: cuatro sistemas y el control
`baseline` (D1), `d2`, `agentes` (D3, modelo base), `agentes_ft` (D4) y `control_icl` (mismos datos como ejemplos en el prompt, pesos base). Primero una prueba corta.

In [ ]:
!python scripts/evaluate_d4.py --limit 2 --configs agentes_ft

In [ ]:
!python scripts/evaluate_d4.py --resume
!cat results/deliverable4/por_pregunta.md

## 4. Demo del video
Mensajes entre agentes en vivo: primero con el Planificador base y después con el ajustado, sobre la misma entrada. `q12` es el caso de falla del D3; `q11-p` es una paráfrasis externa.

In [ ]:
!python scripts/run_agents.py --id q12 --compare --no-baseline

In [ ]:
!python scripts/run_agents.py --id q11-p --compare --no-baseline

## 5. Descargar resultados
Se suben al repositorio respetando las carpetas (`results/deliverable4/`, `docs/deliverable4_results.tex`).

In [ ]:
import shutil, os
staging = "/content/entrega_d4"
shutil.rmtree(staging, ignore_errors=True)
shutil.copytree("results/deliverable4", f"{staging}/results/deliverable4",
                ignore=shutil.ignore_patterns("*.safetensors"))  # los pesos están en Hugging Face
os.makedirs(f"{staging}/docs", exist_ok=True)
shutil.copy("docs/deliverable4_results.tex", f"{staging}/docs/")
shutil.make_archive("/content/deliverable4_resultados", "zip", staging)
from google.colab import files
files.download("/content/deliverable4_resultados.zip")